# Insurance Claim Propensity Modeling

**Recruiter-facing end-to-end analysis · Imbalanced binary classification · Python 3.12/3.13**

> The tuned random forest selected by training PR-AUC reaches untouched-test ROC-AUC 0.800, PR-AUC 0.618, and balanced accuracy 0.709.

## Executive summary

**Objective:** Compare and calibrate claim-propensity models using training-only selection and an untouched test set.

**Data:** 3,000 policy rows with demographic, agency, product, destination, duration, sales, commission, channel, and claim fields.

**Verified result:** The tuned random forest selected by training PR-AUC reaches untouched-test ROC-AUC 0.800, PR-AUC 0.618, and balanced accuracy 0.709.

**Decision supported:** Understand ranking and threshold trade-offs while keeping claim decisions under human governance.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** An insurance analytics or operations team.

**Decision:** Understand ranking and threshold trade-offs while keeping claim decisions under human governance.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '04-insurance-claim-prediction'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.13
pandas            2.2.3
NumPy             2.3.5
SciPy            1.17.0
scikit-learn      1.8.0
Matplotlib       3.10.8

Project: 04-insurance-claim-prediction


## 4. Data provenance and scope

Bundled in the original repository; the upstream sampling frame, collection process, and redistribution terms are not documented.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

                file  size_mb           sha256
insurance_claims.csv    0.183 989cd34d3505baf1


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


insurance_claims.csv: 10 columns
 Age Agency_Code          Type Claimed  Commision Channel  Duration  Sales      Product Name Destination
  48         C2B      Airlines      No       0.70  Online         7   2.51   Customised Plan        ASIA
  36         EPX Travel Agency      No       0.00  Online        34  20.00   Customised Plan        ASIA
  39         CWT Travel Agency      No       5.94  Online         3   9.90   Customised Plan    Americas
  36         EPX Travel Agency      No       0.00  Online         4  26.00 Cancellation Plan        ASIA
  33         JZI      Airlines      No       6.30  Online        53  18.00       Bronze Plan        ASIA


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 203 lines
Functions: _metrics, _pipeline, run_analysis


## 7. Methodology and hypotheses

Duplicate leakage control, mixed-type pipelines, dummy baseline, logistic/CART/random-forest/boosting/MLP comparison, randomized tuning, stratified CV, calibration, threshold selection, PR-AUC, and permutation importance.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_04_insurance_claim_prediction", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 63.92 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (10 fields)
      column   dtype  missing_count  missing_percent  unique_values
         Age   int64              0              0.0             70
 Agency_Code  object              0              0.0              4
        Type  object              0              0.0              2
     Claimed  object              0              0.0              2
   Commision float64              0              0.0            324
     Channel  object              0              0.0              2
    Duration   int64              0              0.0            257
       Sales float64              0              0.0            380
Product Name  object              0              0.0              5
 Destination  object              0              0.0              3


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'model_comparison.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'The tuned random forest selected by training PR-AUC reaches untouched-test ROC-AUC 0.800, PR-AUC 0.618, and balanced accuracy 0.709.')

Primary evidence: model_comparison.csv, shape=(7, 6)
              model  cv_pr_auc_mean  cv_pr_auc_std  cv_roc_auc_mean  cv_balanced_accuracy_mean                selection_stage
random_forest_tuned          0.6822            NaN              NaN                        NaN randomized_search_8_candidates
      random_forest          0.6715         0.0368           0.8109                     0.7411                      candidate
logistic_regression          0.6710         0.0580           0.8103                     0.7445                      candidate
  gradient_boosting          0.6634         0.0489           0.8084                     0.6991                      candidate
                mlp          0.6539         0.0795           0.7851                     0.6804                      candidate
      decision_tree          0.6238         0.0512           0.7926                     0.7211                      candidate
   dummy_prevalence          0.3193         0.0006           0.50

## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])


VALIDATION
{
  "train_rows": 2145,
  "untouched_test_rows": 716,
  "test_fraction": 0.25,
  "selection_cv": "4-fold stratified CV on training data only",
  "optimization_metric": "average precision (PR-AUC)",
  "random_seed": 42
}

TUNING
{
  "model": "random_forest",
  "search": "8-candidate randomized search",
  "best_parameters": {
    "model__n_estimators": 500,
    "model__min_samples_leaf": 8,
    "model__max_features": "sqrt",
    "model__max_depth": 10
  }
}


## 12. Visual evidence

### Best Model Confusion Matrix

![best_model_confusion_matrix](../reports/figures/best_model_confusion_matrix.png)

### Insurance Model Evidence

![insurance_model_evidence](../reports/figures/insurance_model_evidence.png)

## 13. Business interpretation

The tuned random forest selected by training PR-AUC reaches untouched-test ROC-AUC 0.800, PR-AUC 0.618, and balanced accuracy 0.709.

The correct action is to use this result as evidence for **Understand ranking and threshold trade-offs while keeping claim decisions under human governance.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

Educational model only; never use it to approve, deny, price, or investigate an insurance claim.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                       artifact  size_kb       sha256
reports/figures/best_model_confusion_matrix.png     28.0 df615cb6cf09
   reports/figures/insurance_model_evidence.png    202.1 88ba93b9b381
                           reports/metrics.json      5.2 be846eceb9d9
                      reports/original_tree.dot      2.3 ef37c5b802d5
                reports/tables/data_quality.csv      0.3 5f7f13e7c88d
            reports/tables/model_comparison.csv      0.7 56ca2f6cfe5e
      reports/tables/permutation_importance.csv      0.5 b9fa19483822
          reports/tables/threshold_analysis.csv      1.2 4bdeeb4f1502


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed compare and calibrate claim-propensity models using training-only selection and an untouched test set. using duplicate leakage control, mixed-type pipelines, dummy baseline, logistic/cart/random-forest/boosting/mlp comparison, randomized tuning, stratified cv, calibration, threshold selection, pr-auc, and permutation importance. The final verified conclusion is: **The tuned random forest selected by training PR-AUC reaches untouched-test ROC-AUC 0.800, PR-AUC 0.618, and balanced accuracy 0.709.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/04-insurance-claim-prediction/src/analysis.py
python scripts/execute_notebooks.py --project 04-insurance-claim-prediction
```